# 연금 RAG 에이전트 — Colab

로컬 `pension_agent` 와 **파이프라인 동작이 동일**합니다.
경로·비밀키·디바이스를 읽어 오는 방식만 코랩에 맞췄습니다.

| | 로컬 | 코랩 |
|---|---|---|
| 설정값 | `.env` 파일 | 🔑 Secrets → 환경변수 |
| 데이터 경로 | `PENSION_DATA_DIR` | 후보 경로 자동 탐색 |
| 벡터 아티팩트 | `cache/index_*/` | 구글 드라이브 (세션이 끊겨도 유지) |
| 디바이스 | CPU | **GPU 자동 감지** |

### 시작 전에 두 가지

**① GPU 켜기** — 런타임 → 런타임 유형 변경 → 하드웨어 가속기 **T4 GPU**
(없어도 돌아가지만 임베딩이 수십 분 걸립니다)

**② API 키 등록** — 왼쪽 사이드바 🔑 → `CLOVA_API_KEY` 추가 → **노트북 액세스** 켜기
(질문·질의세트 실행에 필요합니다. 키 없이도 벡터 빌드와 검색 점검은 됩니다)
### 두 단계로 나뉩니다

| 단계 | 하는 일 | 비용 | 언제 |
|---|---|---|---|
| **빌드** `index.build_index` | 코퍼스 → 벡터 아티팩트 | 무겁다 (모델 2.2GB + 7,939건 인코딩) | 데이터가 바뀔 때만 |
| **서빙** `run(...)` | 아티팩트 읽어서 검색·답변 | 가볍다 | 매번 |

아티팩트를 드라이브에 두면 **다음 세션부터 빌드를 건너뜁니다.**
코퍼스가 바뀌면 지문이 달라져 자동으로 감지되고, 그때만 다시 빌드하면 됩니다.


---
## 0. 프로젝트 올리기

셋 중 편한 방법 하나만 쓰세요. 이미 올렸다면 이 셀은 경로만 확인합니다.

In [ ]:
# ① 로컬에서 pension_agent_colab 폴더를 zip 으로 압축해 업로드하는 방법
#    (파일 탭에 끌어다 놓아도 됩니다)
import os, sys, zipfile, glob
from pathlib import Path

PROJECT = None
for c in ["/content/pension_agent_colab", "/content/pension_agent",
          "/content/drive/MyDrive/pension_agent_colab"]:
    if Path(c, "config.py").exists():
        PROJECT = c
        break

if PROJECT is None:
    zips = glob.glob("/content/*.zip")
    if zips:
        print(f"압축 해제: {zips[0]}")
        with zipfile.ZipFile(zips[0]) as z:
            z.extractall("/content")
        for c in glob.glob("/content/**/config.py", recursive=True):
            PROJECT = str(Path(c).parent)
            break

if PROJECT is None:
    print("프로젝트를 찾지 못했습니다. 아래 중 하나를 하세요:\n"
          "  · pension_agent_colab.zip 을 파일 탭에 업로드한 뒤 이 셀 재실행\n"
          "  · 또는 구글 드라이브에 올려 두고 drive.mount() 후 재실행")
else:
    os.chdir(PROJECT)
    sys.path.insert(0, PROJECT)
    print(f"작업 폴더 → {PROJECT}")

---
## 1. 환경 준비

의존성 설치 · 드라이브 마운트(캐시 영속화) · 키 확인 · 데이터 위치 탐색을 한 번에 합니다.

In [ ]:
import colab_setup
colab_setup.setup()          # 드라이브를 안 쓰려면 use_drive=False

In [ ]:
# 키가 반영됐는지 확인
#
# config 는 import 시점에 환경변수를 읽으므로, 이미 import 한 뒤 키를 넣으면
# reload 없이는 반영되지 않는다.
#
#   import os
#   os.environ["CLOVA_API_KEY"] = "nv-..."
#   import colab_setup; colab_setup.setup(reload=True)   # ← reload 필수

import config

print("CLOVA 키:", "있음" if config.CLOVA_API_KEY else "없음 (질문 실행 불가)")
print("벡터 필수:", config.REQUIRE_VECTORS,
      "— False 로 두면 아티팩트 없이 BM25 단독으로 돕니다")


---
## 2. 코퍼스 점검 (LLM 호출 0회)

무엇이 로드됐는지, 필터 축이 제대로 잡혔는지 먼저 확인합니다.

In [ ]:
import json
from index import corpus_stats, fund_names

s = corpus_stats()
print(json.dumps(s, ensure_ascii=False, indent=2))
print(f"\n펀드 {len(fund_names())}개 — 질의분석 프롬프트에 주입됩니다")
print("예시:", fund_names()[:3])

In [ ]:
# 검색 동작 확인 — 계층 쿼터가 자리를 어떻게 나누는지
#
# ★ 예전에는 scope 필터가 코퍼스 절반을 **지웠다**. 지금은 지우지 않고
#   top-k 안의 자리를 나눈다. 판정이 틀려도 정답이 사라지지 않는 게 핵심이다.
from collections import Counter

import config
from index import get_index

idx = get_index()
L = config.PRODUCT_INTENT_LEVELS
cases = [
    ("IRP 중도인출이 가능한 사유는?",            L["제도"], "사실조회"),
    ("퇴직연금 적립금의 위험자산 투자한도는?",    L["제도"], "사실조회"),
    ("세액공제 한도와 담을 채권형 펀드 알려줘",   L["혼합"], "혼합"),
    ("퇴직연금 IRP에 가입 가능한 채권형 펀드는?", L["상품"], "추천"),
]
for q, aff, qt in cases:
    f = {"product_affinity": aff}
    k = 24 if aff >= 0.6 else 8
    hits = idx.search(q, top_k=k, filters=f, 질문유형=qt)
    nd = sum(1 for h in hits if h["chunk"]["tier"] == "doc")
    dist = Counter(f"{h['chunk']['tier']}/{h['chunk']['chunk_type']}" for h in hits)
    print()
    print(f"[관여도 {aff:.2f}] {q}")
    print(f"        후보 {len(idx.candidates(f)):,}건 → {len(hits)}건  "
          f"(규정 {nd} / 상품 {len(hits) - nd})  {dict(dist)}")
    for h in hits[:3]:
        c = h["chunk"]
        print(f"          {h['score']:.3f} {c['chunk_id'][:26]:26s} "
              f"{(c.get('section') or c['text'])[:40]}")


---
## 3. 임베딩 인덱스 만들기

`intfloat/multilingual-e5-large` (1024차원). **한 번만** 돌리면 됩니다 —
드라이브에 캐시가 남아 다음 세션에서는 즉시 로드됩니다.

- GPU: 1~2분 / CPU: 수십 분
- E5 는 접두사가 필수라 문서에 `passage: `, 질의에 `query: ` 를 자동으로 붙입니다
- 메모리가 부족하면 가드가 막고 BM25 단독으로 떨어집니다(프로세스는 안 죽습니다)

In [ ]:
# [빌드 1/3] 임베딩 모델이 로드되는지 1건으로 확인 (키 불필요)
!python -m index.build_index --probe


In [ ]:
# [빌드 2/3] 코퍼스 → 벡터 아티팩트
#
# 이미 최신이면 모델을 로드조차 하지 않고 즉시 끝난다.
# 캐시를 드라이브에 뒀다면 두 번째 세션부터는 여기서 "이미 최신입니다" 가 뜬다.
!python -m index.build_index


In [ ]:
# 벡터가 인덱스에 붙었는지 확인 (하이브리드 검색이 켜졌는가)
from index import reset_index, get_index
reset_index()                     # 새로 만든 .npy 를 다시 읽는다
idx = get_index()
print("벡터 인덱스:", None if idx.vectors is None else idx.vectors.shape)

---
## 4. 질문하기

`run()` 이 안전검사 → 질의분석 → 라우팅 → 분해 → (서브질문별 병렬 RAG) →
취합 → 검증 → 합성까지 한 번에 돕니다.

In [ ]:
from utils import render_trace
from workflows import run

SEP = "─" * 68


def ask(question, profile=None, show_trace=True):
    r = run(question, user_profile=profile or {}, qkey=question[:24])
    if show_trace:
        print(render_trace(r.get("trace", [])))
    print()
    print(SEP)
    print("답변")
    print(SEP)
    print(r.get("final_answer") or "(답변 없음)")

    u = r.get("usage") or {}
    total = u.get("total_tokens") or (u.get("in_tokens", 0) + u.get("out_tokens", 0))
    ms = u.get("ms", u.get("total_ms", 0))
    print()
    print(SEP)
    print(f"토큰 {total:,} (입력 {u.get('in_tokens', 0):,} + "
          f"출력 {u.get('out_tokens', 0):,}) | "
          f"LLM {u.get('calls', 0)}회 | {ms / 1000:.1f}초")
    return r


In [ ]:
# 상품 질문 — 관여도가 올라가고 top_k 가 24로 넓어진다.
# 규정문서 자리는 그대로 보장되므로 세제 근거가 사라지지 않는다.
_ = ask("연금저축 세액공제 받으면서 담을 채권형 펀드 추천해줘")


In [ ]:
# 한 질문 안에서 관여도가 갈리는 경우 (①② 운용규제=제도, ③ 상품)
_ = ask("퇴직연금 IRP로 ETF랑 펀드를 굴리려는데 어떻게 구성해야 할까?")


In [ ]:
# 계산 질문 — 숫자는 코드가 만든다 (data/tax_tables.json)
_ = ask("총급여 6000만원인데 연금저축 400만원 IRP 300만원 냈으면 세액공제 얼마인가요?")

---
## 5. 성능 평가

두 갈래로 나뉩니다. **채점은 이 노트북에서 하지 않습니다** — 정답 조건(골드셋)과
채점기는 평가 담당이 소유하고, 여기서는 **실행 기록만** 만들어 넘깁니다.

| | 무엇을 | 산출물 | 비용 |
|---|---|---|---|
| **A. 검색 지표** | 검색이 정답 청크를 가져오는가 | `*.json` (recall·MRR·hit) | LLM 0회 |
| **B. 골드셋 실행** | 골드셋 문항을 파이프라인에 태운다 | **`*.compact.jsonl`** ← 평가 담당에게 전달 | 문항당 8~15회 |

검색이 정답을 못 가져오면 그 뒤 무엇을 고쳐도 정답이 안 나오므로 **A 를 먼저** 봅니다.

> `!python -m eval.runner` 가 아니라 **파이썬 함수로 부릅니다.** 셸로 부르면 호출마다
> 새 프로세스가 떠서 코퍼스·벡터 아티팩트·임베딩 모델을 통째로 다시 올립니다.
> 조건을 바꿔 여러 번 돌리는 실험에서는 그 준비 작업이 실행 시간의 대부분입니다.

In [ ]:
# [점검] 골드셋 형식 — LLM 0회
#
# 실행 뒤에 형식 오류를 발견하면 그 실행이 통째로 버려진다. 먼저 돌린다.
# product.jsonl 이 아직 없으면 경고만 나오고 policy 트랙만 로드된다.
from eval import gold

items = gold.load_goldset(strict=False)
print(gold.stats(items))
print(gold.validate(items) or "형식 OK")

In [ ]:
# [A] 검색만 — LLM 호출 0회, 비용 없음
#
# 관여도 중립(0.5) vs 관여도 반영 두 조건을 같이 재서 자리 배분의 효과를 본다.
from eval import runner

res = runner.run_retrieval(tag="colab_retrieval")

for mode, m in res["report"]["by_mode"].items():
    print(f"{mode:10s} recall={m['recall']:.3f}  mrr={m['mrr']:.3f}  hit={m['hit']:.3f}")

In [ ]:
# [경로·비용] 파이프라인을 끝까지 돌린다 (LLM 심판은 제외)
#
# llm_judge=True 로 두면 정확도·충실도까지 재지만 비용이 크게 는다.
# 정확도 판정은 평가 담당의 골드셋 채점이 맡으므로 여기서는 끈다.
res = runner.run_pipeline(llm_judge=False, tag="colab_pipeline")

In [ ]:
# 질문별 토큰 — 어느 질문이 비싼지, 어느 단계가 먹는지
t = res["report"]["tokens"]
print(f"질문당 평균 {t['avg_total_per_question']:,} 토큰 / 최대 {t['max_total_per_question']:,}")
print(f"전체 합계 {t['sum_total']:,}\n")

for qid, u in sorted(t["per_question"].items()):
    print(f"  {qid}  {u['total_tokens']:>7,} 토큰  ({u['calls']}회 호출)")

print("\n모델별:", t["overall"].get("by_model", {}))

In [ ]:
# [B] 골드셋 실행 — 문항을 파이프라인에 태워 실행 기록만 남긴다 (채점 없음)
#
# 조건마다 tag 를 다르게 준다. 실험 조건(top_k·bm25_w·WORKFLOW_SETTINGS)은
# compact.jsonl 첫 줄 _meta 에 자동으로 박히므로 나중에 추적할 수 있다.
#
#   track="policy"   제도 트랙만. product.jsonl 이 들어오면 track="" 로 둔다
#   split="dev"      튜닝용만. 최종 확인은 split="holdout" 으로 한 번만
gold_run = runner.run_gold(split="dev", track="policy", tag="baseline")

print("\n넘길 파일:", gold_run["compact"])

In [ ]:
# 평가 담당에게 넘길 파일 — compact.jsonl 하나만 전달하면 된다
import glob
import json
import os

from google.colab import files

hits = sorted(glob.glob("eval/results/*.compact.jsonl"), key=os.path.getmtime)
if not hits:
    print("(아직 없음 — 위 셀을 먼저 실행하세요)")
else:
    for h in hits:                       # 조건별로 여러 개가 쌓인다
        with open(h, encoding="utf-8") as f:
            meta = json.loads(f.readline()).get("_meta", {})
        print(f"{os.path.basename(h):46s} {meta.get('n', '?')}건  "
              f"tag={meta.get('tag')}  split={meta.get('split')}")
    files.download(hits[-1])             # 가장 최근 것

---
## 6. Ablation — 구성요소 기여도 재기

조건을 바꿔가며 `tag` 를 다르게 주고 돌린 뒤 `eval/results/*.json` 을 비교합니다.
각 파일에 `settings`·`params`·`tokens` 가 함께 저장되므로 어떤 조건이었는지
나중에 추적할 수 있습니다.

> 검색 가중치는 `index/store.py` 가 **검색할 때마다** `config` 에서 읽습니다.
> 그래서 값을 바꾸고 바로 다시 돌리면 됩니다 — 프로세스를 새로 띄울 필요가 없습니다.
> 실험이 끝나면 원래 값으로 되돌려 두세요.

In [ ]:
# 예: 하이브리드 가중치 비교 (LLM 0회)
import config
from eval import runner

원래 = (config.BM25_WEIGHT, config.VEC_WEIGHT)
결과 = {}

for bm25, vec in [(0.25, 0.75), (0.35, 0.65), (0.50, 0.50)]:
    config.BM25_WEIGHT, config.VEC_WEIGHT = bm25, vec
    print(f"\n{'=' * 60}\nBM25 {bm25} : 벡터 {vec}\n{'=' * 60}")
    r = runner.run_retrieval(tag=f"w_bm{bm25}")
    결과[bm25] = r["report"]["by_mode"]["affinity"]

config.BM25_WEIGHT, config.VEC_WEIGHT = 원래      # 원상 복구
print("\n되돌림:", 원래)

In [ ]:
# 결과 모아 보기
print(f"{'bm25_w':>8s} {'recall':>8s} {'mrr':>8s} {'hit':>8s}")
for bm25, m in sorted(결과.items()):
    print(f"{bm25:8.2f} {m['recall']:8.3f} {m['mrr']:8.3f} {m['hit']:8.3f}")

---
## 7. 결과 내려받기

평가 결과와 임베딩 캐시를 로컬로 가져갑니다.
(드라이브를 마운트했다면 이미 드라이브에 있습니다)

In [ ]:
import shutil
from google.colab import files

shutil.make_archive("/content/eval_results", "zip", "eval/results")
files.download("/content/eval_results.zip")

---
## 8. 질의 세트 78문항 실행 (자체 평가)

상품 트랙 41 + 제도·세제·절차 트랙 37 = **78문항**을 돌리고, 성공/실패를
판정할 수 있는 기록을 남깁니다.

⚠️ 전체 실행은 LLM 호출이 많습니다(문항당 10~15회). 먼저 `--limit 5` 로
배선을 확인한 뒤 전체를 돌리세요. 중간에 끊기면 `--resume` 으로 이어서 갑니다.

생성물 3종 중 **`.compact.jsonl`** 이 분석용입니다.

In [ ]:
# 먼저 5문항으로 배선 확인
!python -m eval.run_queryset --limit 5 --tag colab_smoke


In [ ]:
# 전체 78문항 (시간·비용이 듭니다)
!python -m eval.run_queryset --tag colab_full

# 트랙·섹션·유형별로 나눠 돌리려면:
#   !python -m eval.run_queryset --track 상품 --tag colab_product
#   !python -m eval.run_queryset --section C --tag colab_classcode
#   !python -m eval.run_queryset --kind 역질문 --tag colab_clarify
#   !python -m eval.run_queryset --tag colab_full --resume   # 중단 지점부터


In [ ]:
# 분석용 기록 내려받기 — 이 파일을 분석에 사용합니다
import glob

from google.colab import files

hits = sorted(glob.glob("eval/results/*.compact.jsonl"))
print(("\n").join(hits) or "(아직 없음 — 위 셀을 먼저 실행하세요)")
if hits:
    files.download(hits[-1])
